# IQN: sample a continuous quantile function

IQN samples $\tau\sim U(0,1)$ and conditions state features on a cosine embedding
$$\phi(\tau)=\operatorname{ReLU}\left(W\,[\cos(\pi k\tau)]_{k=0}^{K-1}+b\right).$$
Here $K$ is the embedding size. The resulting $Z_\tau(s,a)$ can represent any requested quantile level rather than a fixed grid.

## 1. Embed sampled quantile fractions

State features and quantile embeddings interact element-wise before the action head.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

EMBEDDING_DIM = 64
COSINES = 32

class IQN(nn.Module):
    def __init__(self, observations=4, actions=2):
        super().__init__()
        self.features = nn.Sequential(nn.Linear(observations, EMBEDDING_DIM), nn.ReLU())
        self.quantile = nn.Linear(COSINES, EMBEDDING_DIM)
        self.head = nn.Sequential(nn.Linear(EMBEDDING_DIM, 64), nn.ReLU(), nn.Linear(64, actions))
    def forward(self, observations, taus):
        batch, samples = taus.shape
        indices = torch.arange(COSINES, device=taus.device, dtype=taus.dtype)
        cosine = torch.cos(torch.pi * taus[..., None] * indices)
        embedding = torch.relu(self.quantile(cosine))
        features = self.features(observations)[:, None, :]
        return self.head(features * embedding)

network = IQN()

## 2. Sample and average quantiles

The mean over sampled quantiles estimates $Q(s,a)=\mathbb E_	au[Z_	au(s,a)]$.

In [ ]:
observation = torch.tensor([[0.1, -0.2, 0.3, 0.0]])
taus = torch.rand(1, 64)
quantile_values = network(observation, taus)
q_values = quantile_values.mean(dim=1)
print("Sampled Q-values:", q_values.detach().numpy())

## 3. Trace the quantile function

Evaluating an ordered tau grid reveals the learned continuous return curve for each action.

In [ ]:
tau_grid = torch.linspace(0.0, 1.0, 101).unsqueeze(0)
with torch.no_grad(): values = network(observation, tau_grid).squeeze(0).numpy()
plt.plot(tau_grid.squeeze(0), values[:, 0], label="Action 0")
plt.plot(tau_grid.squeeze(0), values[:, 1], label="Action 1")
plt.xlabel("Quantile fraction")
plt.ylabel("Return quantile")
plt.title("IQN quantile functions")
plt.legend(); plt.grid(alpha=0.2); plt.show()